In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

np.random.seed(42)

def matern52_2d(X1, X2, sigma_sq, ls):
    """2D Matérn-5/2 kernel with per-dimension length-scales.
    ls = [ℓ_x, ℓ_y] — anisotropic length-scales.
    """
    # Scale each dimension by its length-scale
    X1_scaled = X1 / np.array(ls)
    X2_scaled = X2 / np.array(ls)

    dist = cdist(X1_scaled, X2_scaled, metric='euclidean')
    sqrt5_r = np.sqrt(5) * dist
    return sigma_sq * (1 + sqrt5_r + 5 * dist**2 / 3) * np.exp(-sqrt5_r)

# Generate a grid for visualization
nx, ny = 50, 50
x_grid = np.linspace(0, 4, nx)
y_grid = np.linspace(0, 4, ny)
xx, yy = np.meshgrid(x_grid, y_grid)
X_grid = np.column_stack([xx.ravel(), yy.ravel()])  # Shape: (2500, 2)

sigma_sq = 100.0

# Compare isotropic vs anisotropic
configs = {
    'Isotropic\nℓ_x = ℓ_y = 1.0 km': [1.0, 1.0],
    'Mild anisotropy\nℓ_x=2.0, ℓ_y=1.0': [2.0, 1.0],
    'Strong anisotropy\nℓ_x=3.0, ℓ_y=0.5': [3.0, 0.5],
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, ls) in zip(axes, configs.items()):
    K = matern52_2d(X_grid, X_grid, sigma_sq, ls) + 1e-4 * np.eye(len(X_grid))
    sample = np.random.multivariate_normal(np.zeros(len(X_grid)), K)

    im = ax.contourf(xx, yy, sample.reshape(nx, ny), levels=20, cmap='RdYlGn')
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Easting (km)')
    ax.set_ylabel('Northing (km)')
    ax.set_aspect('equal')
    plt.colorbar(im, ax=ax, label='RMR')

plt.suptitle('Isotropic vs Anisotropic GP Prior Draws (2D)', fontsize=13)
plt.tight_layout()
plt.savefig("anisotropy_comparison.png", dpi=150)
plt.show()

print("=== OBSERVATIONS ===")
print("Isotropic:        Circular blobs. Equal correlation in all directions.")
print("Mild anisotropy:  Elliptical blobs, stretched horizontally.")
print("Strong anisotropy: Bands/layers! Looks like geological strata.")
print()
print("→ Strong anisotropy (ℓ_x >> ℓ_y) produces LAYERED spatial fields.")
print("→ This is exactly what real geological cross-sections look like.")
print("→ In PyMC: pm.gp.cov.Matern52(input_dim=2, ls=[ℓ_x, ℓ_y])")

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

np.random.seed(42)

# === MOCK BOREHOLE DATA ===
# 8 boreholes at random map coordinates within a 4km × 4km study area
n_boreholes = 8
x_coords = np.array([0.5, 1.2, 2.0, 0.8, 3.0, 3.5, 1.5, 2.8])  # Easting (km)
y_coords = np.array([0.5, 1.5, 0.8, 2.8, 1.0, 2.5, 3.2, 3.0])  # Northing (km)
rmr_values = np.array([55, 42, 48, 30, 52, 35, 28, 32])           # RMR

# Assemble into 2D input matrix
X_obs = np.column_stack([x_coords, y_coords])  # Shape: (8, 2)
y_obs = rmr_values.astype(float)

print(f"=== 2D BOREHOLE DATA ===")
print(f"Study area: 4 km × 4 km")
print(f"Boreholes:  {n_boreholes}")
print(f"Input shape: {X_obs.shape} (n_samples, 2)")
for i in range(n_boreholes):
    print(f"  BH-{i+1}: ({x_coords[i]:.1f}, {y_coords[i]:.1f}) km  →  RMR = {rmr_values[i]}")

# Zero-center for HSGP (Day 3 lesson!)
X_center = X_obs.mean(axis=0)
X_centered = X_obs - X_center
print(f"\nCenter: ({X_center[0]:.2f}, {X_center[1]:.2f})")
print(f"Centered range: [{X_centered.min():.2f}, {X_centered.max():.2f}]")

# Visualize borehole locations
fig, ax = plt.subplots(figsize=(7, 7))
scatter = ax.scatter(x_coords, y_coords, c=rmr_values, cmap='RdYlGn',
                      s=200, edgecolors='black', linewidth=2, zorder=5,
                      vmin=20, vmax=60)
for i in range(n_boreholes):
    ax.annotate(f'BH-{i+1}\nRMR={rmr_values[i]}',
                (x_coords[i], y_coords[i]),
                textcoords="offset points", xytext=(12, 8),
                fontsize=8, fontweight='bold')
ax.set_xlabel('Easting (km)', fontsize=12)
ax.set_ylabel('Northing (km)', fontsize=12)
ax.set_title('Borehole Locations & RMR Values', fontsize=13)
ax.set_xlim(0, 4)
ax.set_ylim(0, 4)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='RMR')
plt.tight_layout()
plt.savefig("borehole_map.png", dpi=150)
plt.show()

In [ ]:
# === 2D HSGP MODEL WITH ANISOTROPIC KERNEL ===
with pm.Model() as spatial_model:

    # LENGTH-SCALES: one per dimension (anisotropic)
    # ℓ_x (Easting): how far does RMR correlate horizontally?
    # ℓ_y (Northing): how far does RMR correlate vertically?
    # We expect ℓ_x ≥ ℓ_y (horizontal correlation ≥ vertical)
    ell_x = pm.InverseGamma("ell_x", alpha=3, beta=3)
    ell_y = pm.InverseGamma("ell_y", alpha=3, beta=3)

    # AMPLITUDE
    eta = pm.HalfCauchy("eta", beta=15)

    # NOISE
    sigma = pm.HalfNormal("sigma", sigma=5)

    # KERNEL: 2D Matérn-5/2 with SEPARATE length-scales
    # input_dim=2: two spatial dimensions
    # ls=[ℓ_x, ℓ_y]: per-dimension length-scales (ARD kernel)
    cov_func = eta**2 * pm.gp.cov.Matern52(input_dim=2, ls=[ell_x, ell_y])

    # HSGP: m=[15, 15] = 225 basis functions total
    # c=1.5 with centered data
    gp = pm.gp.HSGP(
        m=[15, 15],     # 15 basis functions per dimension
        c=1.5,          # Boundary extension
        cov_func=cov_func,
    )

    # Prior on latent GP
    f = gp.prior("f", X=X_centered)

    # Likelihood
    y_ = pm.Normal("y_obs", mu=f, sigma=sigma, observed=y_obs)

print(spatial_model)

# === SAMPLE ===
with spatial_model:
    trace_2d = pm.sample(
        draws=1000, tune=1000, chains=4,
        nuts_sampler="numpyro",
        target_accept=0.9,
        random_seed=42,
    )

# === DIAGNOSTICS ===
print("\n=== POSTERIOR SUMMARY ===")
print(az.summary(trace_2d, var_names=["ell_x", "ell_y", "eta", "sigma"]))

az.plot_trace(trace_2d, var_names=["ell_x", "ell_y", "eta", "sigma"])
plt.suptitle("2D GP Hyperparameter Traces", fontsize=14)
plt.tight_layout()
plt.savefig("2d_gp_traces.png", dpi=150)
plt.show()

# Check anisotropy: is ℓ_x > ℓ_y?
ell_x_post = trace_2d.posterior["ell_x"].values.flatten()
ell_y_post = trace_2d.posterior["ell_y"].values.flatten()
aniso_ratio = ell_x_post / ell_y_post
print(f"\nAnisotropy ratio ℓ_x/ℓ_y: {aniso_ratio.mean():.2f} "
      f"[{np.percentile(aniso_ratio, 5):.2f}, {np.percentile(aniso_ratio, 95):.2f}]")
print("→ Ratio > 1: horizontal correlation is longer (expected for layered geology)")
print("→ Ratio ≈ 1: approximately isotropic (no directional preference)")